### **1. Monthly sales trend (With growth rate)**

In [0]:
%sql
WITH monthly_sales AS (
    SELECT
        YEAR(InvoiceDate) AS yr,
        MONTH(InvoiceDate) AS mn,
        SUM(Quantity * Price) AS revenue
    FROM retail
    GROUP BY yr, mn
)
SELECT
    yr,
    mn,
    revenue,
    LAG(revenue) OVER (ORDER BY yr, mn) AS prev_month_revenue,
    (revenue - LAG(revenue) OVER (ORDER BY yr, mn)) / 
    LAG(revenue) OVER (ORDER BY yr, mn) AS growth_rate
FROM monthly_sales
ORDER BY yr, mn

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

**Monthly Sales Conclusion**
1. Strong year-end spike (Oct–Nov peak)
2. Sales drop sharply in early months (Jan–Feb weak demand)
3. Consistent seasonal pattern across years
4. 2011 shows highest peak → demand growth over time
5. December drop in 2011 → incomplete data or anomaly

**Monthly Growth rate cycle Conclusion**
1. Growth is highly volatile → no stable expansion
2. Positive spikes followed by sharp drops → demand shocks
3. Similar patterns across years → recurring seasonal behavior
4. Strong positive growth before peak months → buildup effect
5. Sharp negative in Dec (esp 2011) → anomaly or incomplete data

### **2. Top Products by Revenue (Ranking)**

In [0]:
%sql
SELECT
    Description,
    SUM(Quantity * Price) AS revenue,
    RANK() OVER (ORDER BY SUM(Quantity * Price) DESC) AS rank
FROM retail
GROUP BY Description


-- Below results shows out of 5k products, their are only top 10 products that contributed 6 figure revenue accross all two year span.

### **3. Top Product sold per Month (window + partition)**

In [0]:
%sql
WITH monthly_product AS (
    SELECT
        YEAR(InvoiceDate) AS yr,
        MONTH(InvoiceDate) AS mn,
        Description,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY yr, mn, Description
)
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY yr, mn ORDER BY total_qty DESC) AS rnk
    FROM monthly_product
) sub
WHERE rnk = 1

### **4. Revenue Contribution % of Each Product**

In [0]:
%sql
SELECT
    Description,
    SUM(Quantity * Price) AS revenue,
    SUM(Quantity * Price) / SUM(SUM(Quantity * Price)) OVER () AS revenue_share
FROM retail
GROUP BY Description
ORDER BY revenue DESC

### **5. Peak Sales Day of Week**

In [0]:
%sql
SELECT
    DAYOFWEEK(InvoiceDate) AS day,
    SUM(Quantity) AS total_sales
FROM retail
GROUP BY day
ORDER BY total_sales DESC

### **6. Country-wise Top Product**

In [0]:
%sql

WITH country_product AS (
    SELECT
        Country,
        Description,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Country, Description
)
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY Country ORDER BY total_qty DESC) AS rnk
    FROM country_product
)
WHERE rnk = 1

### **7. Sales Spike Detection (vs average)**

In [0]:
%sql
WITH daily_sales AS (
    SELECT
        DATE(InvoiceDate) AS dt,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY dt
)
SELECT
    dt,
    total_qty,
    AVG(total_qty) OVER () AS avg_sales,
    CASE 
        WHEN total_qty > 2 * AVG(total_qty) OVER () THEN 'Spike'
        ELSE 'Normal'
    END AS sales_flag
FROM daily_sales

-- Compared each day’s sales against overall average to detect unusually high demand days, Spike logic here is 2X of avg daily sales (quantity)